In [1]:
import os

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
from langchain.chat_models import init_chat_model
# Grog model

model = init_chat_model("groq:qwen/qwen3-32b")


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title")
    year:int=Field(description="The year movie was released")
    director:str=Field(description="The director")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about movie Inception")
response


Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [4]:
#message output  with parsed 
class Movie(BaseModel):
    title:str=Field(..., description="The year movie was released")
    year:int=Field(...,description="The year movie was released")
    director:str=Field(..., description="The director")
    rating:float=Field(..., description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("Provide details about movie Inception")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires title, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the title is obviously "Inception". The year it was released was 2010. The director is Christopher Nolan. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb does list it at 8.8. So I have all the required parameters. I\'ll structure the tool call with these details. Make sure the types are correct: title as a string, year as an integer, director as a string, and rating as a number. Everything looks good. Time to put it into the JSON format specified.\n', 'tool_calls': [{'id': '1akxfb3kb', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, 

In [ ]:
#nested structure
class Actor(BaseModel):
    name:str
    roel:str

class Movie(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None= Field(description="Budget in million USD")
    

model_with_structure = model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about movie Inception")
response


Movie(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', roel='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', roel='Arthur'), Actor(name='Ellen Page', roel='Mal'), Actor(name='Tom Hardy', roel='Eames')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

In [11]:
#typedDict

from typing_extensions import TypedDict, Annotated

class Movie(TypedDict):
     title: Annotated[str,..., "The year movie was released"]
     year:  Annotated[int, ...,"The year movie was released"]
     director:Annotated[str, ..., "The director"]
     rating:Annotated[float,..., "The movie rating out of 10"]

model_with_structure = model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about movie avengers")
response


{'director': 'Joss Whedon', 'rating': 8.1, 'title': 'Avengers', 'year': 2012}

In [14]:
#nested structure
class Actor(TypedDict):
    name:str
    roel:str

class Movie(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None= Field(description="Budget in million USD")
    

model_with_structure = model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about movie Avengers")
response


{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'roel': 'Iron Man'},
  {'name': 'Chris Evans', 'roel': 'Captain America'},
  {'name': 'Mark Ruffalo', 'roel': 'Hulk'},
  {'name': 'Chris Hemsworth', 'roel': 'Thor'},
  {'name': 'Scarlett Johansson', 'roel': 'Black Widow'},
  {'name': 'Jeremy Renner', 'roel': 'Hawkeye'}],
 'genres': ['Action', 'Science Fiction', 'Superhero'],
 'title': 'Avengers',
 'year': 2012}

In [ ]:
model_with_structure.profile

AttributeError: 'RunnableSequence' object has no attribute 'profile'

In [16]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [ ]:
#dataclasses
#pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

In [20]:
class ContactInfo(BaseModel):
    """contact info"""
    name:str=Field(description="name")
    eamil:str=Field(description="eamil")
    phone:str=Field(description="phone")

   




In [ ]:
agent = create_agent(model="groq:qwen/qwen3-32b", response_format=ContactInfo)

In [22]:
result= agent.invoke({"messages":[{"role":"user", "content": "Extract info from John Doe, john@gmail.com, (555)1234567"}]})
result

{'messages': [HumanMessage(content='Extract info from John Doe, john@gmail.com, (555)1234567', additional_kwargs={}, response_metadata={}, id='2272a782-9bd4-4503-aae0-f42d360600dc'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user provided a string with John Doe\'s contact information. I need to extract the name, email, and phone number. Let me start by breaking down the input.\n\nThe input is "John Doe, john@gmail.com, (555)1234567". The name is straightforward: John Doe. The email is clearly "john@gmail.com". The phone number is "(555)1234567". \n\nWait, the phone number has parentheses and a space. The function parameters require the phone as a string. Should I format it as is or remove the parentheses? The example in the tool\'s parameters doesn\'t specify formatting, so I\'ll include it as provided. \n\nCheck the required parameters: name, eamil, phone. All three are present here. The email is correctly formatted. The phone number includes parenthes

In [24]:
result["structured_response"]

ContactInfo(name='John Doe', eamil='john@gmail.com', phone='(555)1234567')

In [25]:
#TypedDict

from typing_extensions import TypedDict

class ContactInfo(TypedDict):
    """contact info"""
    name:str
    eamil:str
    phone:str

agent = create_agent(model="groq:qwen/qwen3-32b", response_format=ContactInfo)

result= agent.invoke({"messages":[{"role":"user", "content": "Extract info from John Doe, john@gmail.com, (555)1234567"}]})
result["structured_response"]


{'name': 'John Doe', 'eamil': 'john@gmail.com', 'phone': '(555)1234567'}

In [26]:
#Dataclasses
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """contact info"""
    name:str
    eamil:str
    phone:str

agent = create_agent(model="groq:qwen/qwen3-32b", response_format=ContactInfo)

result= agent.invoke({"messages":[{"role":"user", "content": "Extract info from John Doe, john@gmail.com, (555)1234567"}]})
result["structured_response"]

ContactInfo(name='John Doe', eamil='john@gmail.com', phone='(555)1234567')